# Ultimate judgement

**COSC2753 Assignment 2 - Fashion Intelligence System**

Four models were built over the same catalogue of fashion product photographs:
item type, season, gender and usage, and visual search. Each was selected and
reported inside its own notebook, against its own held-out split. This notebook
asks the question none of them can answer alone.

## The judgement

> **The system is fit to deploy against catalogue-style product photography, and
> is not fit to deploy against unconstrained user photographs without an explicit
> confidence gate. The binding constraint is not model capacity and it is not
> input resolution. It is the gap between the flat-lay images the models were
> trained on and the images a user actually submits.**

Three claims, and all three are testable. The first says the models clear a useful
bar on the distribution they were built for. The second says they degrade on a
different distribution, and that the degradation is a property of the training
data rather than of the networks. The third rules out the explanation that was
assumed for most of this project's life, which is that 60x80 pixels were the
ceiling. Sections 1, 2 and 7 take them in turn.

## What this notebook computes

Nothing. Every figure below is read from an artefact written by one of the four
task notebooks or by a script in `scripts/`. That is deliberate: a synthesis that
recomputes its inputs can disagree with the notebooks it summarises, and this one
cannot. It runs in seconds and it is stale only if an artefact is stale.

In [1]:
# ============================================
# CELL 1 - Setup
# ============================================

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path.cwd() if (Path.cwd() / "artifacts").exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR))

ARTIFACTS = PROJECT_DIR / "artifacts"
OUTPUTS = PROJECT_DIR / "outputs"
EVALUATION = OUTPUTS / "evaluation"

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)


def load_json(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else {}


def load_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def majority_class_weighted_f1():
    """Weighted F1 of always predicting the most common training class.

    Reads the shared split and the cleaned metadata rather than any model
    output, so this stays consistent with the notebook's rule of reporting
    figures rather than recomputing model results.
    """
    from sklearn.metrics import f1_score

    splits = load_csv(PROJECT_DIR / "A2_FashionDataset" / "processed" / "splits_120x160.csv")
    meta = load_csv(PROJECT_DIR / "A2_FashionDataset" / "processed" / "clean_train_metadata.csv")
    if splits.empty or meta.empty:
        return np.nan

    known = set(task1_checkpoint["class_names"])
    joined = splits.merge(meta[["id", "articleType"]], on="id")
    joined = joined[joined["articleType"].isin(known)]
    train = joined[joined["split"] == "train"]["articleType"]
    test = joined[joined["split"] == "test"]["articleType"]
    if train.empty or test.empty:
        return np.nan

    majority = train.mode()[0]
    predicted = np.full(len(test), majority)
    return 100 * f1_score(test, predicted, average="weighted", zero_division=0)


# Task 1 is read from the final deployed checkpoint, not the legacy summary.
task1_checkpoint_path = ARTIFACTS / "task1_120x160" / "task1_120x160_onecycle_best.pt"
task1_checkpoint = torch.load(task1_checkpoint_path, map_location="cpu", weights_only=False)
task1_test = task1_checkpoint["test_metrics"]
task1 = {
    "test_accuracy_deployed": task1_test["accuracy"] * 100,
    "test_weighted_f1_deployed": task1_test["weighted_f1"] * 100,
    "test_macro_f1_deployed": task1_test["macro_f1"] * 100,
    "test_balanced_accuracy_deployed": task1_test["balanced_acc"] * 100,
    "classes_after_drop": task1_checkpoint["num_classes"],
    "checkpoint": str(task1_checkpoint_path),
    # The majority-class baseline is a property of the LABEL DISTRIBUTION, not of
    # the model or its input resolution, so it is neither a 60x80 figure nor
    # something the checkpoint needs to carry. Computing it on the same split the
    # checkpoint was scored on keeps Task 1 in the comparison below; leaving it
    # NaN drops the strongest task out of the only cross-task table in the
    # notebook and silently changes which task the cell reports as strongest.
    # It comes out at 5.15 here against 5.27 on the older 60x80 partition, which
    # is the evidence that the split barely moves it.
    "baseline_weighted_f1": majority_class_weighted_f1(),
}
task2 = load_json(ARTIFACTS / "task2" / "task2_season_metrics.json")
task3 = load_json(ARTIFACTS / "task3" / "task3_cnn_summary.json")
task4 = load_json(ARTIFACTS / "task4" / "task4_summary.json")
manifest = load_json(ARTIFACTS / "task4" / "search_manifest.json")

sources = {
    "task 1 final checkpoint": task1_checkpoint,
    "task 2 metrics": task2,
    "task 3 summary": task3,
    "task 4 summary": task4,
    "task 4 manifest": manifest,
}
for name, blob in sources.items():
    print("{:<24s} {}".format(name, "loaded" if blob else "MISSING"))

print("Task 1 final checkpoint: {}".format(task1_checkpoint_path))
print("Task 1 input: {}x{}".format(*task1_checkpoint["image_size_pil"]))
print("Task 4 serves at {} and the encoder is {}background augmented.".format(
    "x".join(str(v) for v in manifest.get("image_size_pil", [])),
    "" if manifest.get("background_augmented") else "NOT ",
))

task 1 final checkpoint  loaded
task 2 metrics           loaded
task 3 summary           loaded
task 4 summary           loaded
task 4 manifest          loaded
Task 1 final checkpoint: C:\Users\LEGION\Documents\jupyter_notebook\Fashion-Intelligence-System\artifacts\task1_120x160\task1_120x160_onecycle_best.pt
Task 1 input: 120x160
Task 4 serves at 120x160 and the encoder is background augmented.


## 1. What each model achieves on the distribution it was built for

The four tasks are not comparable on a single number. A 92-class problem, a
4-class problem, a 5-class and a 4-class problem sharing a trunk, and a ranking
problem have different ceilings and different floors, and a metric that is
flattering to one is punishing to another.

What makes them comparable is the *margin over the naive answer*: how much better
is the model than always predicting the majority class, or than returning random
items? The last column normalises that margin by the room the naive answer leaves,
so a task with a high floor is not credited for the floor.

In [2]:
# ============================================
# CELL 2 - Headline performance, and the margin over doing nothing
# ============================================

rows = [
    {
        "task": "1 - item type",
        "target": "articleType (92 classes)",
        "metric": "weighted F1",
        "model": task1.get("test_weighted_f1_deployed"),
        "naive": task1.get("baseline_weighted_f1"),
    },
    {
        "task": "2 - season",
        "target": "season (4 classes)",
        "metric": "macro F1",
        "model": task2.get("test", {}).get("macro_f1", np.nan) * 100,
        "naive": task2.get("baseline", {}).get("macro_f1", np.nan) * 100,
    },
    {
        "task": "3 - gender",
        "target": "gender (5 classes)",
        "metric": "accuracy",
        "model": task3.get("test_gender_accuracy"),
        "naive": task3.get("baseline_gender_accuracy"),
    },
    {
        "task": "3 - usage",
        "target": "usage (4 classes)",
        "metric": "accuracy",
        "model": task3.get("test_usage_accuracy"),
        "naive": task3.get("baseline_usage_accuracy"),
    },
    {
        "task": "4 - visual search",
        "target": "same articleType in top 10",
        "metric": "P@10",
        "model": task4.get("P@10_unseen"),
        "naive": task4.get("random_baseline_P@10"),
    },
]

headline = pd.DataFrame(rows)
headline["margin"] = headline["model"] - headline["naive"]
headline["headroom closed"] = (
    100 * headline["margin"] / (100 - headline["naive"])
)
headline = headline.round(2)
display(headline)

weakest = headline.loc[headline["headroom closed"].idxmin()]
strongest = headline.loc[headline["headroom closed"].idxmax()]
print("strongest : {:<18s} {:.1f}% of the available headroom closed".format(
    strongest["task"], strongest["headroom closed"]))
print("weakest   : {:<18s} {:.1f}%".format(
    weakest["task"], weakest["headroom closed"]))
print()
print("Every model clears its naive floor by a wide margin, so the first claim of")
print("the judgement holds. The spread between the strongest and the weakest is")
print("the first thing a deployment decision has to price: these are not four")
print("models of equal standing, and section 6 says which is which.")

,task,target,metric,model,naive,margin,headroom closed
0,1 - item type,articleType (92 classes),weighted F1,89.68,5.15,84.53,89.12
1,2 - season,season (4 classes),macro F1,64.32,16.57,47.74,57.23
2,3 - gender,gender (5 classes),accuracy,90.33,54.20,36.13,78.89
3,3 - usage,usage (4 classes),accuracy,90.24,77.11,13.13,57.36
4,4 - visual search,same articleType in top 10,P@10,76.19,5.91,70.28,74.69


strongest : 1 - item type      89.1% of the available headroom closed
weakest   : 2 - season         57.2%

Every model clears its naive floor by a wide margin, so the first claim of
the judgement holds. The spread between the strongest and the weakest is
the first thing a deployment decision has to price: these are not four
models of equal standing, and section 6 says which is which.


## 2. The failure they share, measured twice by two tasks that were not testing each other

Each notebook reports a domain gap of its own. Read together they are plainly the
same gap, and establishing that is the reason this notebook exists.

The two measurements are genuinely independent, and they were taken at
different times. Task 1 measured it by **corrupting its own held-out inputs**:
the same 790 test rows, composited onto textures and jittered, scored by a
classifier. Those rows were produced with the earlier 60x80 checkpoint, before
Task 1's final 120x160 model existed. That dates the
measurement; it does not retire the finding, because what the rows establish is
a property of the **training distribution** - flat-lay catalogue tiles - which
both Task 1 checkpoints share. Task 4 measured it by
**compositing held-out catalogue items onto photographs from Places365
categories the encoder never saw**, scored by a retrieval protocol. Different
task, different architecture, different metric, different corruption family,
built months apart for different reasons.

They agree, and they agree on the magnitude as well as the direction: both models
retain roughly a tenth of their in-domain performance.

In [3]:
# ============================================
# CELL 3 - The same collapse, measured twice, independently
# ============================================

ood = load_csv(EVALUATION / "task1_ood_results.csv")
arms = load_csv(EVALUATION / "task4_background_arms.csv")

gap_rows = []

# Historical Task 1 evidence: these rows predate the final 120x160 deployment.
# They remain useful for the domain-gap argument, but are not the final model.
if not ood.empty:
    historical = ood[ood["checkpoint"] == "task1_cnn.pt"]
    for ingest in sorted(historical["ingest"].unique()):
        subset = historical[historical["ingest"] == ingest]
        clean = float(subset[subset["severity"] == "clean"]["accuracy"].iloc[0])
        worst = float(subset[subset["severity"] == "severe"]["accuracy"].iloc[0])
        gap_rows.append({
            "task": "1 - item type",
            "model": "task1_cnn.pt (historical 60x80)",
            "measured by": "historical corrupted inputs, ingest={}".format(ingest),
            "metric": "accuracy",
            "in-domain": clean,
            "out-of-domain": worst,
        })

# Task 4: the catalogue-only control arm, on clean frames and on photographs.
if not arms.empty:
    control = arms[arms["arm"] == "C_encoder_clean"]
    clean = float(control[control["benchmark"] == "clean"]["P@10"].iloc[0])
    photo = float(control[control["benchmark"] == "photo"]["P@10"].iloc[0])
    gap_rows.append({
        "task": "4 - visual search",
        "model": "arm C, catalogue-only control",
        "measured by": "held-out Places365 backdrops",
        "metric": "P@10",
        "in-domain": clean,
        "out-of-domain": photo,
    })

gap = pd.DataFrame(gap_rows)
gap["drop"] = (gap["out-of-domain"] - gap["in-domain"]).round(2)
gap["retained"] = (100 * gap["out-of-domain"] / gap["in-domain"]).round(1)
display(gap.round(2))

print("The Task 1 rows above are historical 60x80 evidence; the final Task 1")
print("model is the 120x160 checkpoint loaded in Cell 1.")
print("Task 4 provides the independent background-domain comparison.")

,task,model,measured by,metric,in-domain,out-of-domain,drop,retained
0,1 - item type,task1_cnn.pt (historical 60x80),"historical corrupted inputs, ingest=nobg",accuracy,79.11,10.51,-68.60,13.3
1,1 - item type,task1_cnn.pt (historical 60x80),"historical corrupted inputs, ingest=squash",accuracy,87.85,6.58,-81.27,7.5
2,4 - visual search,"arm C, catalogue-only control",held-out Places365 backdrops,P@10,81.64,11.84,-69.80,14.5


The Task 1 rows above are historical 60x80 evidence; the final Task 1
model is the 120x160 checkpoint loaded in Cell 1.
Task 4 provides the independent background-domain comparison.


### The gap is a property of the training distribution, not of the architecture

The strongest evidence that this is a data problem rather than a capacity problem
is that the *same intervention* was applied in both tasks, with no architectural
change in either, and it moved the out-of-domain number by tens of points in both.

- **Task 1 historical evidence** used the earlier 60x80 `ItemTypeCNN` under the
  `webphoto` recipe. Those rows remain useful as development evidence, but they
  are not the final deployed model.
- **Task 4** trained the identical `ImprovedEncoder` with garments composited onto
  Places365 scenes. Same four conv blocks, same 128-d projection, same losses.

The final Task 1 model is the 120x160 checkpoint loaded in Cell 1:
`artifacts/task1_120x160/task1_120x160_onecycle_best.pt`. The historical
`candidate_webphoto.pt` and `task1_cnn.pt` rows below are explicitly retained as
measurement evidence only; neither is a current deployment artefact.

In [4]:
# ============================================
# CELL 4 - The same intervention, in two tasks, changing the data and not the model
# ============================================

exchange = []

# This Task 1 comparison is historical 60x80 evidence. It does not describe
# the final 120x160 checkpoint loaded in Cell 1.
if not ood.empty:
    corrupted = ["mild", "moderate", "severe"]
    for ingest in sorted(ood["ingest"].unique()):
        subset = ood[ood["ingest"] == ingest]

        def score(checkpoint, severities):
            rows = subset[(subset["checkpoint"] == checkpoint)
                          & (subset["severity"].isin(severities))]
            return float(rows["accuracy"].mean())

        base_clean = score("task1_cnn.pt", ["clean"])
        aug_clean = score("candidate_webphoto.pt", ["clean"])
        base_ood = score("task1_cnn.pt", corrupted)
        aug_ood = score("candidate_webphoto.pt", corrupted)
        exchange.append({
            "task": "1 - item type (historical 60x80)",
            "arms": "historical webphoto recipe vs historical 60x80 checkpoint, ingest={}".format(ingest),
            "in-domain change": aug_clean - base_clean,
            "out-of-domain change": aug_ood - base_ood,
        })

significance = load_csv(EVALUATION / "task4_background_arms_significance.csv")
if not significance.empty:
    typed = significance[significance["metric"] == "type@10"]
    clean_delta = float(typed[typed["benchmark"] == "clean"]["delta"].iloc[0])
    photo_delta = float(typed[typed["benchmark"] == "photo"]["delta"].iloc[0])
    exchange.append({
        "task": "4 - visual search",
        "arms": "arm D vs arm C, Places365 backdrops",
        "in-domain change": clean_delta,
        "out-of-domain change": photo_delta,
    })

exchange = pd.DataFrame(exchange)


def exchange_rate(row):
    """Points bought out of domain per point spent in domain."""
    spent = -row["in-domain change"]
    if spent <= 0:
        return "no in-domain cost"
    return "{:.2f} : 1".format(row["out-of-domain change"] / spent)


exchange["exchange rate"] = exchange.apply(exchange_rate, axis=1)
display(exchange.round(2))

print("The Task 1 rows are explicitly historical 60x80 evidence; the final")
print("Task 1 deployment is the 120x160 checkpoint loaded in Cell 1.")
print("Task 4 supplies the current 120x160 background-intervention comparison.")

,task,arms,in-domain change,out-of-domain change,exchange rate
0,1 - item type (historical 60x80),historical webphoto recipe vs historical 60x80...,4.56,20.38,no in-domain cost
1,1 - item type (historical 60x80),historical webphoto recipe vs historical 60x80...,-2.79,22.28,7.99 : 1
2,4 - visual search,"arm D vs arm C, Places365 backdrops",-5.45,43.94,8.06 : 1


The Task 1 rows are explicitly historical 60x80 evidence; the final
Task 1 deployment is the 120x160 checkpoint loaded in Cell 1.
Task 4 supplies the current 120x160 background-intervention comparison.


### The intervention only pays if the representation can absorb it

"Augment the training data" is the wrong lesson to draw from the table above, and
Task 4 is the only task in the project that ran the control needed to show it.

Its 2x2 crosses **model family** with **data treatment**. Arms C and D are the one
network trained two ways; arms A and B are one hand-built descriptor built two
ways - 128-bin HSV plus 108-bin gradient histograms, no learning at all, taken
from the catalogue as it ships (A) or averaged over composited views (B).

The classical row is what makes the notebook able to attribute anything. Without
it, "the augmented encoder beats the clean encoder out of domain" is compatible
with the lesson being about the data alone. Every arm is scored on identical query
frames, so every difference below is paired.

In [5]:
# ============================================
# CELL 5 - The 2x2, and the cross-term that is the finding
# ============================================

classical = load_csv(EVALUATION / "task4_classical_clustering_arms_v3.csv")

if not classical.empty and not arms.empty:
    exact = classical[classical["index"] == "exact"]
    grid = pd.concat([
        exact.pivot_table(index="arm", columns="benchmark", values="P@10"),
        arms.pivot_table(index="arm", columns="benchmark", values="P@10"),
    ])
    order = ["clean", "hard", "photo", "wild", "wildphoto"]
    grid = grid[[c for c in order if c in grid.columns]]
    display(grid.round(2))

    def trade(before, after):
        return (grid.loc[after, "clean"] - grid.loc[before, "clean"],
                grid.loc[after, "photo"] - grid.loc[before, "photo"])

    classical_cost, classical_gain = trade("A_classical_clean", "B_classical_bgaug")
    encoder_cost, encoder_gain = trade("C_encoder_clean", "D_encoder_bgaug")

    print("the same intervention, in two representations")
    print("  hand-built descriptor : {:+6.2f} clean for {:+6.2f} photo   ratio {:.2f} : 1".format(
        classical_cost, classical_gain, classical_gain / -classical_cost))
    print("  learned encoder       : {:+6.2f} clean for {:+6.2f} photo   ratio {:.2f} : 1".format(
        encoder_cost, encoder_gain, encoder_gain / -encoder_cost))
    print()
    print("Averaging a fixed histogram over backdrops buys invariance by destroying")
    print("the signal: in a composite the garment is a minority of the pixels, so")
    print("the average is mostly background. The encoder can learn to ignore the")
    print("backdrop instead of averaging it away. Arm B is not a failed experiment,")
    print("it is the control that makes the encoder's number mean something.")
    print()
    print("Arm C is the sharpest result here. It is the best model in the project")
    print("on clean catalogue images ({:.2f} against the deployed {:.2f}) and near".format(
        grid.loc["C_encoder_clean", "clean"], grid.loc["D_encoder_bgaug", "clean"]))
    print("the worst on a photograph ({:.2f} against {:.2f}). Selecting on a clean".format(
        grid.loc["C_encoder_clean", "photo"], grid.loc["D_encoder_bgaug", "photo"]))
    print("benchmark alone would have shipped it.")

benchmark,clean,hard,photo,wild,wildphoto
arm,,,,,
A_classical_clean,69.31,16.39,10.17,13.17,8.58
B_classical_bgaug,35.74,28.56,20.93,18.79,13.13
C_encoder_clean,81.64,9.10,11.84,9.21,6.36
D_encoder_bgaug,76.19,56.32,55.78,53.07,51.62


the same intervention, in two representations
  hand-built descriptor : -33.57 clean for +10.76 photo   ratio 0.32 : 1
  learned encoder       :  -5.45 clean for +43.94 photo   ratio 8.06 : 1

Averaging a fixed histogram over backdrops buys invariance by destroying
the signal: in a composite the garment is a minority of the pixels, so
the average is mostly background. The encoder can learn to ignore the
backdrop instead of averaging it away. Arm B is not a failed experiment,
it is the control that makes the encoder's number mean something.

Arm C is the sharpest result here. It is the best model in the project
on clean catalogue images (81.64 against the deployed 76.19) and near
the worst on a photograph (11.84 against 55.78). Selecting on a clean
benchmark alone would have shipped it.


## 3. Why the two tasks then made opposite deployment decisions

Task 4 deployed the background-augmented model. Task 1 did not. The measurement
pointed the same way in both, so the difference is not a disagreement about
evidence. It is a difference in **what each model is actually asked**.

- Task 1's deliverable is a prediction for `styles_prediction_template.csv`, which
  is 5,829 catalogue tiles from the same photographic pipeline as the training
  set. Its inputs are in domain by construction, so the in-domain cost of
  augmentation is a real cost and the out-of-domain gain buys nothing that is
  graded.
- Task 4's deliverable is a search box in an application that accepts an upload.
  Its inputs are whatever a user has on their phone, so the trade runs the other
  way.

Same evidence, opposite decisions, and both are right. This is the concrete
content of the phrase "fit for purpose": the model that wins is a function of the
input distribution the model will meet, and neither notebook could have worked
that out from its own held-out split.

It is also the reason `DEPLOYMENT_WEIGHT = 3.0` in Task 4's epoch-selection rule
is defensible rather than arbitrary. It was set to value out-of-domain performance
at 3:1 before anything had been measured; the paired 2x2 puts the real exchange
rate well above that, so the rule was conservative rather than wrong.

## 4. Independent evaluation

Two forms of independence are worth separating, because they support different
things.

**A reference implementation trained by someone else.** Task 1 was compared
against a frozen ImageNet backbone with a logistic probe on the same split, which
sets a bar the from-scratch network had to clear.

**Data the models never saw, from outside the dataset entirely.** The 31
photographs in `A2_FashionDataset/input_images` were collected from the open web:
mixed formats, cluttered backgrounds, worn garments, several items per frame, and
some non-clothing. They share no provenance with the supplied catalogue, which is
what makes them the only check available on whether the composited benchmarks
this project relies on are honest.

In [6]:
# ============================================
# CELL 6 - Against a pretrained reference, and the state of the real photographs
# ============================================

reference = load_json(ARTIFACTS / "task1" / "pretrained_reference.json")
if reference:
    comparison = pd.DataFrame([
        {"model": "Task 1 CNN, trained from scratch",
         "test accuracy": task1.get("test_accuracy_deployed"),
         "test weighted F1": task1.get("test_weighted_f1_deployed"),
         "test macro F1": task1.get("test_macro_f1_deployed")},
        {"model": "frozen ImageNet backbone, same split",
         "test accuracy": reference["test"]["accuracy"],
         "test weighted F1": reference["test"]["weighted_f1"],
         "test macro F1": reference["test"]["macro_f1"]},
    ]).round(2)
    display(comparison)
    delta = (task1.get("test_weighted_f1_deployed", 0)
             - reference["test"]["weighted_f1"])
    print("from-scratch network is {:+.2f} weighted F1 against the reference".format(delta))
    print()

real = load_csv(EVALUATION / "task4_real_photo_arms.csv")
if not real.empty:
    display(real[["arm", "labelled", "P@1", "P@10",
                  "top1 similarity", "similarity drop", "ingestion declined"]])

    # The same two encoders on composited frames, so the proxy can be checked
    # against the thing it stands in for.
    composited = arms.pivot_table(index="arm", columns="benchmark", values="P@10")
    proxy = pd.DataFrame({
        "arm": ["arm C, catalogue-only", "arm D, deployed"],
        "real photographs (n=23)": [
            float(real[real["arm"] == "white only"]["P@10"].iloc[0]),
            float(real[real["arm"] == "white + backgrounds"]["P@10"].iloc[0]),
        ],
        "composited `photo` (n=2000)": [
            float(composited.loc["C_encoder_clean", "photo"]),
            float(composited.loc["D_encoder_bgaug", "photo"]),
        ],
    })
    proxy["proxy overstates by"] = (
        proxy["composited `photo` (n=2000)"] - proxy["real photographs (n=23)"])
    display(proxy.round(2))

    ratio_real = proxy.loc[1, "real photographs (n=23)"] / proxy.loc[0, "real photographs (n=23)"]
    ratio_comp = (proxy.loc[1, "composited `photo` (n=2000)"]
                  / proxy.loc[0, "composited `photo` (n=2000)"])
    print("augmented over control, real photographs : {:.1f}x".format(ratio_real))
    print("augmented over control, composited       : {:.1f}x".format(ratio_comp))

,model,test accuracy,test weighted F1,test macro F1
0,"Task 1 CNN, trained from scratch",90.23,89.68,73.11
1,"frozen ImageNet backbone, same split",75.59,76.40,58.40


from-scratch network is +13.27 weighted F1 against the reference



,arm,labelled,P@1,P@10,top1 similarity,similarity drop,ingestion declined
0,white only,23.0,8.70,8.70,0.6702,0.1062,1
1,white + backgrounds,23.0,26.09,23.04,0.7249,0.0715,1


,arm,real photographs (n=23),composited `photo` (n=2000),proxy overstates by
0,"arm C, catalogue-only",8.70,11.84,3.14
1,"arm D, deployed",23.04,55.78,32.74


augmented over control, real photographs : 2.6x
augmented over control, composited       : 4.7x


### The composited benchmark is an optimistic proxy

The photographs were labelled by hand on 2026-09-09, so for the first time this
project has a retrieval score measured on real uploads rather than on
constructions. 23 of the 31 carry a catalogue `articleType`; the other 8 are two
stickers, a set of sleep masks no catalogue class covers, a six-item flat-lay and
four street-style shots where two garments are co-equal. Those are marked `none`
and excluded, rather than forced into a label that is not true or counted as
misses.

**The ordering survives the move to real photographs and the level does not.**
The augmented encoder beats the control by 2.6x here against 4.7x on composites,
so the finding in section 2 holds. But it scores 23.04 where the composited
`photo` benchmark says 55.78 for the same encoder on the same task. Pasting a
catalogue cutout onto a Places365 scene is **easier** than a real upload: the
cutout is already segmented, centred, lit like a catalogue tile, and unoccluded.

So every out-of-domain number in this project is an **upper bound**, and `photo`
and `wildphoto` should be read that way. This does not weaken the judgement, it
sharpens it: the gap between catalogue performance and real-world performance is
*larger* than the composited benchmarks say.

**And the proxy is not equally optimistic for both arms**, which is the part that
matters for how the 2x2 is read. It overstates the deployed encoder by 32.7
points and the control by 3.1. That is not a coincidence: a composite is built
the same way the augmented encoder's training data was built, from a segmented
cutout pasted onto a scene, so it rewards exactly the invariance that training
installed. The `photo` benchmark uses held-out Places365 *categories*, so it is
not circular in the way the old `hard_metrics` figure was, but it is not neutral
either. The honest reading of section 2 is therefore that background augmentation
is a large real gain whose size on real uploads is **2.6x, not 4.7x** - the
direction is safe, the multiplier was flattered by the measuring instrument.

Three limits travel with 23.04 and belong beside it wherever it is quoted.
**n=23**, so the sampling error is roughly +/-10 points and no precise claim rests
on it. **The labels were produced by a vision model rather than a person** and
are unverified; they agree with Task 1's own top-1 on only 5 of 23, so they are
at least not circular, but a human pass would make them ground truth rather than
a second opinion. And **1 of the 31 defeats ingestion entirely**, finding no
plausible garment mask and falling back to a centre crop.

What needs no labels at all, and is worth keeping beside the scores: the two
encoders' top-10 sets **overlap by 6.5%** and agree on the single best match for
**1 of 31** photographs. Changing the training data does not nudge the answer on
a real photo, it replaces it.

## 5. The degradation is detectable at inference time

A system that fails on out-of-distribution input but cannot tell that it is
failing is unusable in production. A system that fails and *knows* it is failing
can decline, ask for a better photograph, or route to a human.

Two label-free signals were already available and are surfaced by the API: the
top-1 similarity, and the coherence of the returned set, which is the share of the
top-K agreeing with the top result's type.

In [7]:
# ============================================
# CELL 7 - Confidence signals over the whole unlabelled test set
# ============================================

test_summary = load_json(OUTPUTS / "task4_test_summary.json")

if test_summary:
    print("images scored               : {:,}".format(test_summary["images"]))
    print("mean top-1 similarity       : {:.3f}".format(test_summary["mean_top1_similarity"]))
    print("mean coherence              : {:.3f}".format(test_summary["mean_coherence"]))
    print("answered confidently        : {:.1%}".format(test_summary["confident_share"]))
    print("ingestion fell back         : {:.1%}".format(test_summary["ingest_fell_back_share"]))
    print()
    print("About seven in ten catalogue-style images are answered confidently under")
    print("thresholds calibrated to sit between the two populations. The remainder")
    print("are the ones a gate is for.")
    print()
    print("Task 3 carries the equivalent for classification. As trained it reported")
    print("98.6% mean confidence on gender against 90.3% accuracy; one temperature")
    print("per head, fitted on validation, brings expected calibration error from")
    print("8.24 to 1.23 without changing a single prediction. Fitted on catalogue")
    print("images, so it makes in-domain confidence mean what it says and does not")
    print("fix out-of-distribution overconfidence.")
    print()
    print("temperatures in the served checkpoint : {}".format(task3.get("temperature")))

images scored               : 5,829
mean top-1 similarity       : 0.826
mean coherence              : 0.743
answered confidently        : 70.2%
ingestion fell back         : 37.6%

About seven in ten catalogue-style images are answered confidently under
thresholds calibrated to sit between the two populations. The remainder
are the ones a gate is for.

Task 3 carries the equivalent for classification. As trained it reported
98.6% mean confidence on gender against 90.3% accuracy; one temperature
per head, fitted on validation, brings expected calibration error from
8.24 to 1.23 without changing a single prediction. Fitted on catalogue
images, so it makes in-domain confidence mean what it says and does not
fix out-of-distribution overconfidence.

temperatures in the served checkpoint : {'gender': 5.8289, 'usage': 5.9429}


## 6. Where the system fails, stated plainly

Four limitations are load-bearing enough that a deployment decision has to
account for them. None of them is a bug, and three of them have been measured
directly enough to say what kind of limitation they are.

**Season is weakly determined.** Task 2 closes the smallest share of its available
headroom of the four, because the target is only loosely a property of the image.
A navy shirt is not visibly autumn.

**The long tail stays weak.** Task 1's final 120x160 checkpoint still has a
substantial weighted-F1 to macro-F1 gap: the headline is carried by common
classes while rare ones remain unreliable. The 32 classes dropped for having too
few examples are a deliberate scope reduction, not a solved problem.

**Some labels are not in the pixels.** The family-level accuracy and symmetric
confusion analysis retained elsewhere in this notebook is historical Task 1
60x80 evidence. It remains useful for diagnosing catalogue label ambiguity, but
its old 88.29%/87.87% figures are not the final 120x160 headline. Task 3's
`usage` head is outscored on `Sports` recall by nothing more than the network's
own image evidence: an oracle told the exact `articleType` reaches 54.9% where
the network reaches 64.8%. And 39% of Task 3's gender errors involve `Unisex`, a
label that records how a seller listed an item.

**One vector per image.** Task 4 embeds a whole frame, so a photograph of a person
wearing several garments is answered as though it were one item. Region search
mitigates this and does not remove it.

In [8]:
# ============================================
# CELL 8 - The long tail and the weak heads, quantified
# ============================================

print("Task 1 final 120x160 checkpoint")
print("  accuracy {:.2f}   weighted F1 {:.2f}   macro F1 {:.2f}   balanced accuracy {:.2f}".format(
    task1.get("test_accuracy_deployed", float("nan")),
    task1.get("test_weighted_f1_deployed", float("nan")),
    task1.get("test_macro_f1_deployed", float("nan")),
    task1.get("test_balanced_accuracy_deployed", float("nan"))))
print("  classes kept {}".format(task1.get("classes_after_drop")))
print("  checkpoint {}".format(task1.get("checkpoint")))
print("  historical train-gap and seed statistics are not copied from the legacy 60x80 summary")

print()
print("Task 3")
print("  gender  accuracy {:.2f}   macro F1 {:.2f}   gap {:.2f}".format(
    task3.get("test_gender_accuracy", float("nan")),
    task3.get("test_gender_macro_f1", float("nan")),
    task3.get("test_gender_accuracy", 0) - task3.get("test_gender_macro_f1", 0)))
print("  usage   accuracy {:.2f}   macro F1 {:.2f}   gap {:.2f}".format(
    task3.get("test_usage_accuracy", float("nan")),
    task3.get("test_usage_macro_f1", float("nan")),
    task3.get("test_usage_accuracy", 0) - task3.get("test_usage_macro_f1", 0)))
print("  exact match {:.2f}   measured noise floor {:.2f}".format(
    task3.get("test_exact_match", float("nan")),
    task3.get("noise_floor", float("nan"))))

print()
print("Task 4")
print("  P@10 {:.2f}   colour@10 {:.2f}   both@10 {:.2f}".format(
    task4.get("P@10_unseen", float("nan")),
    task4.get("colour@10_unseen", float("nan")),
    task4.get("both@10_unseen", float("nan"))))
print("  gallery {:,} items, {}-d, {:.2f} ms per query".format(
    task4.get("gallery_size", 0), task4.get("embedding_dim", 0),
    task4.get("ms_per_query", float("nan"))))

print()
print("The Task 1 headline above is read directly from the final 120x160 checkpoint.")
print("Historical 60x80 diagnostics are retained only where explicitly labelled.")

Task 1 final 120x160 checkpoint
  accuracy 90.23   weighted F1 89.68   macro F1 73.11   balanced accuracy 72.84
  classes kept 92
  checkpoint C:\Users\LEGION\Documents\jupyter_notebook\Fashion-Intelligence-System\artifacts\task1_120x160\task1_120x160_onecycle_best.pt
  historical train-gap and seed statistics are not copied from the legacy 60x80 summary

Task 3
  gender  accuracy 90.33   macro F1 76.37   gap 13.96
  usage   accuracy 90.24   macro F1 82.66   gap 7.58
  exact match 81.59   measured noise floor 0.67

Task 4
  P@10 76.19   colour@10 55.23   both@10 41.98
  gallery 38,571 items, 128-d, 0.02 ms per query

The Task 1 headline above is read directly from the final 120x160 checkpoint.
Historical 60x80 diagnostics are retained only where explicitly labelled.


## 7. Resolution was the obvious explanation, and it was measured and rejected

For most of this project's life the assumed ceiling was the input: at 60x80 a
lipstick and a deodorant occupy the same few hundred pixels, so of course the
models are limited. It is a good hypothesis and it is testable, and every test run
against it has come back negative.

| test | result |
|---|---|
| Task 3 retrained at 120x160 with a fifth conv block | within +/- 0.2 on every test metric, at four times the compute |
| Task 1 retrained at 120x160 | +0.39 weighted F1 on the rows neither model trained on, CI [-0.55, +1.35], and 0.9 *worse* on macro F1 |
| Task 4 encoder at 120x160 vs 60x80 | ties on `clean`, `hard` and `wild` against a +/- 1.1 floor; only `photo` and `wildphoto` clear it |

Three tasks, three architectures, four times the pixels, and the result is a tie
each time. Set against the tens of points that changing the training distribution
moved in section 2, the conclusion is not close.

This matters for the judgement because it changes the recommendation. The earlier
version of this notebook closed by saying the single change that would move the
conclusion most is higher-resolution source images. That is now measured, and it
is wrong. Resolution buys a fraction of a point. **What the evidence supports is
that the binding constraint is the training distribution**, and the cheapest
remaining move is not more pixels but more variety in the frames the models are
trained on.

The one place resolution did earn its cost is Task 4's colour metrics, which rose
on all five benchmarks at 120x160. That is the only measured resolution win in the
project, and it is why Task 4 alone runs at 120x160.

## 8. The operating policy this implies

The judgement is conditional, so the conditions have to be written down.

| Input | Decision | Basis |
|---|---|---|
| Catalogue-style product photograph | Serve all four predictions | section 1 |
| User photograph, confident signals | Serve, with the confidence shown | section 5 separates the populations |
| User photograph, low confidence | Decline, or ask for a plain-background photo | section 2 shows the collapse is severe, not gradual |
| Several garments in frame | Return per-region results, not one answer | section 6 |
| Season, in any context | Present as a suggestion, never as a label | section 1, weakest of the four |

Three consequences of that table are already built into the system rather than
recommended by it.

**The deployed encoder is the background-augmented one despite being worse in
domain.** Section 2 prices that decision: 5.45 points of clean P@10 for 43.94 out
of domain.

**Region masking happens before the top-k, not after it.** The upper band of a
photograph of socks had all 120 of its nearest neighbours in `Personal Care` under
this encoder, so a post-filter had nothing to keep and returned perfume bottles.
Masking to wearable rows first guarantees a band ranks among garments however far
down the nearest one sits.

**The prior correction is applied to the graded submission and not to uploads.**
The label-shift correction is calibrated on the graded population; an upload is
not drawn from it, so applying it there would be a bias with no basis.

## 9. The judgement, restated

The system meets a useful bar on the distribution it was built for. Item type,
gender and usage are all well above their naive floors and are usable as catalogue
tooling. Season is the weakest and should be presented as a suggestion rather than
a label. Visual search returns a same-type item in roughly three of four top-10
slots on catalogue queries.

It does not meet that bar on unconstrained photographs, and the shortfall is large
enough that presenting the same interface for both inputs would be misleading. It
is also larger than the composited benchmarks report: measured on real uploads
rather than constructions, retrieval scores 23.04 where the `photo` benchmark says
55.78, so every out-of-domain figure quoted here is an upper bound.

The mitigation is not a better architecture, and it is not more pixels. Both were
tried and measured, and both came back as ties. Changing the training distribution
is the only intervention in this project that moved an out-of-domain number by
more than the noise, and section 2 shows it doing so twice, in two tasks, at
roughly the same price. What remains after that is a constrained input and an
explicit gate.

**Deploy against catalogue photography. Gate everything else. The lever is the
data, not the network and not the resolution.**

## 10. What this notebook does not establish

Stated so a reader does not have to infer it.

**One seed per arm, everywhere the arms matter.** Task 4's 2x2 intervals are
paired query-sampling error over 2,000 queries, not seed variance, which is
unmeasured for retrieval. The 23 to 44 point out-of-domain gaps dwarf any
plausible seed effect; the 5.45 point in-domain cost does not, and a second seed
on arms C and D is the only thing that would sharpen it. Task 1's `webphoto`
comparison is likewise a single run per recipe.

**Neither comparison in section 2 can be re-run from a fresh clone.** Task 1's
`candidate_webphoto.pt` was deleted after measurement and is not in the
repository at all. Task 4's arm C still exists on the machine that trained it, at
`artifacts/task4_120x160/task4_encoder_none_seed42.pt`, but that directory is
gitignored, so a clone does not have it either. Both sets of numbers are recorded
in `outputs/evaluation/`; the checkpoints behind them are not, and reproducing
either means retraining.

Arm C is a **control and must never be promoted.** It is the best model in the
project on the clean benchmark, `scripts/promote_task4_encoder.py` would serve it
without complaint, and the only thing on disk that distinguishes it is
`background_augmented: False` in its checkpoint.

**The real-photo score rests on 23 rows and on labels this project produced
itself.** `A2_FashionDataset/input_images_labels.csv` was filled in on 2026-09-09
by a vision model, not by a person, and has not been checked by one. The sampling
error at n=23 is roughly +/-10 points. The labels agree with Task 1's own top-1 on
only 5 of 23, so they are not circular, but "an independent second opinion" is a
fair description of them and "ground truth" is not. Three of the 23 sit on the
`Casual Shoes` / `Sports Shoes` boundary, which is the project's largest single
confusion pair, so that judgement call moves the score more than any other.

**Splits are duplicate-free, not leakage-free.** Product variants shot in the same
frame are near-identical, differently named, and byte-distinct, so both guards
miss them. At least 1.5% of Task 3's held-out rows have such a twin in train. The
net effect on the headline numbers is inside the noise threshold, but the correct
phrase is "no duplicate images", never "no leakage".